# 25B The agent loop
A budget guard, a trace with stated rationales, and replays that are byte-identical. The model here is a deterministic stand-in; yours goes through the record/replay client.

In [2]:
import sys, json, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "creditwatch").exists())
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd, yaml
from creditwatch import (approvals, audit, catalog, channel, clock, econ, evals, heartbeat, llm, loop,
                         memory, panel, policy, signals, skills, standin, workspace)
P = yaml.safe_load(open(ROOT / "params.yaml"))
FX = ROOT / P["data"]["fixture_dir"]
USERS = json.load(open(FX / "users.json"))
MANDATES = json.load(open(ROOT / P["data"]["mandates"]))
REG = json.load(open(ROOT / P["data"]["registry"]))
PANEL = panel.load_panel(ROOT / P["data"]["panel"])
M = P["heartbeat"]["materiality"]
NOW = clock.clock()                       # LANTERN_CLOCK, never the system clock
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 60)
print("fixture mode:", P["data"]["use_fixture"], "| clock:", NOW.isoformat())

fixture mode: True | clock: 2026-10-15T06:00:00-04:00


In [3]:
s = channel.login("priya", "priya-fixture-token", "mandate_a", USERS)
tools = standin.read_tools(s, audit.Ledger(clock=clock.clock), PANEL, MANDATES, M)
PLAN = [{"tool": "mandate_profile", "args": {"scope": "mandate_a"}, "rationale": "the client's positions"},
        {"tool": "bdc_signals", "args": {"scope": "public"}, "rationale": "what changed this release"},
        {"tool": "bdc_holdings", "args": {"scope": "public"}, "rationale": "terms behind each signal"}]
text, trace = loop.run_agent(standin.FixtureModel(PLAN, standin.note_writer), tools, "Draft the client note.", clock.clock)
print(pd.DataFrame(trace)[["step", "tool", "action_class", "rationale", "args_hash", "observation_ref", "usd"]].fillna(""))
print("\n" + text)

   step             tool action_class                  rationale     args_hash observation_ref      usd
0     1  mandate_profile         READ     the client's positions  b5e40423f722    9788a32e0c2b  0.00128
1     2      bdc_signals         READ  what changed this release  7adee52be77e    e178cb953a6e  0.00128
2     3     bdc_holdings         READ   terms behind each signal  7adee52be77e    052ef18fd6c4  0.00128
3     4                                                                                         0.00128

Hotel Senior Lending: Pinecrest Learning Group Inc., First Lien Term Loan maturity extended from 2029-09-30 to 2031-09-30
Hotel Senior Lending: Brightwater Fitness Holdings LLC, First Lien Term Loan marked down 5.6%
Hotel Senior Lending: Oakmont Home Services LLC, First Lien Term Loan PIK up from 1.00% to 3.00%
Juliett Lending BDC: Crestline Roofing Holdings LLC marked down 7.7%
Juliett Lending BDC: Foxglove Specialty Pharma Inc. PIK up from 0.00% to 2.50%
PIK (watch): First

In [4]:
forever = standin.FixtureModel([{"tool": "bdc_signals", "args": {"scope": "public"}}] * 50)
_, t1 = loop.run_agent(forever, tools, "loop", clock.clock, max_steps=P["budget"]["max_steps_per_task"])
_, t2 = loop.run_agent(standin.FixtureModel(PLAN, usd=0.30), tools, "pricey", clock.clock, usd_cap=P["budget"]["usd_cap_per_task"])
print("a model that loops:", t1[-1]["stop"], "after", len(t1) - 1, "steps")
print("an expensive one:  ", t2[-1]["stop"], "at step", t2[-1]["step"], f"(spent ${sum(r['usd'] for r in t2):.2f} against a ${P['budget']['usd_cap_per_task']:.2f} cap)")

a model that loops: max_steps after 8 steps
an expensive one:   budget_usd at step 2 (spent $0.60 against a $0.50 cap)


In [5]:
# Record once, replay forever: same request, same answer, no cost.
import shutil
cache = ROOT / ".cache" / "nb25_llm"
shutil.rmtree(cache, ignore_errors=True)               # start empty, so the demo reruns the same way
calls = []
fake_provider = lambda req: calls.append(1) or {"text": "Juliett: one markdown, one PIK increase.", "usd": 0.00128}
req = {"model": "small", "messages": [{"role": "user", "content": "Summarize Juliett."}]}
print(llm.ReplayClient(cache, "record", call=fake_provider)(req), "| provider calls:", len(calls))
print(llm.ReplayClient(cache, "replay")(req), "| provider calls:", len(calls))
a = json.dumps(loop.run_agent(standin.FixtureModel(PLAN, standin.note_writer), tools, "x", clock.clock)[1], sort_keys=True)
b = json.dumps(loop.run_agent(standin.FixtureModel(PLAN, standin.note_writer), tools, "x", clock.clock)[1], sort_keys=True)
print("two replays give byte-identical traces:", a == b)

{'text': 'Juliett: one markdown, one PIK increase.', 'usd': 0.00128} | provider calls: 1
{'text': 'Juliett: one markdown, one PIK increase.', 'usd': 0.00128} | provider calls: 1
two replays give byte-identical traces: True


**Your turn.** Put your real model behind `ReplayClient` (record mode once, replay in tests and CI), route the small model to triage and materiality, and show one dev task replayed twice with identical traces (Part 2).

In [6]:
import json
import sys

sys.path.insert(0, str(ROOT))
from services.agent.run import verify

bundle = ROOT / "evals" / "recordings" / "dev-01"

# Actually replay the recorded task twice—no OpenAI calls.
report = verify(bundle)

first = json.loads((bundle / "replay-1.json").read_text())
second = json.loads((bundle / "replay-2.json").read_text())

print("Task:", first["task_id"])
print("Two replay files identical:",
      (bundle / "replay-1.json").read_bytes()
      == (bundle / "replay-2.json").read_bytes())
print("Both match the recording:", report["matches_recording"])
print("API calls during replay:", report["provider_calls_during_replay"])

display(pd.DataFrame(first["trace"]))
print("\nReplayed client note:\n")
print(first["final_response"])

Task: dev-01
Two replay files identical: True
Both match the recording: True
API calls during replay: 0


,action_class,args_hash,observation_ref,rationale,run_id,step,tokens_in,tokens_out,tool,ts,usd
0,READ,b5e40423f722,969d0761da01,Must verify the active mandate first and read mandate_pr...,dev-01,1,2051,71,mandate_profile,2026-10-15T06:00:00-04:00,0.006193
1,READ,7adee52be77e,dd18e70e273f,Need public BDC signal evidence before drafting the mand...,dev-01,2,2140,53,bdc_signals,2026-10-15T06:00:00-04:00,0.006145
2,NaN,NaN,NaN,Finished client note draft using the supplied mandate pr...,dev-01,3,2641,553,NaN,2026-10-15T06:00:00-04:00,0.014897



Replayed client note:

Harbor State Employees’ Retirement System monthly credit note draft

Coverage: 3 of 3 held BDCs reviewed. Signals were provided for 2 of 3 held BDCs, with no new signals for FIXTURE GOLF CAPITAL CORP.

Harbor State’s held BDCs showed five reportable credit signals in the latest release set.

FIXTURE HOTEL SENIOR LENDING
- Pinecrest Learning Group Inc., first lien term loan, maturity extended to 2031-09-30 from 2029-09-30, a 24-month extension that exceeds the 6-month threshold. Source: accession 0009000012-26-000121, row 105; prior row 505.
- Brightwater Fitness Holdings LLC, first lien term loan, marked down to 93.5 from 99.0, a 5.5-point decline that meets the materiality threshold. Source: accession 0009000012-26-000121, row 101; prior row 501.
- Oakmont Home Services LLC, first lien term loan, PIK increased to 3.0 from 1.0, a 2.0-point rise above the 0.5-point threshold. Source: accession 0009000012-26-000121, row 104; prior row 504.

FIXTURE JULIETT LENDING